# Creating Null Models for Novel Domain Architectures

Given the results from the cancer fusion gene analysis, we will establish some null models to determine the effect sizes and if we are observing false discoveries in the data.

**Note** this will take a long time (2+ hrs) to run because of how many trials are being run for the TCGA equivalent size.

In [ ]:
import pandas as pd
import numpy as np
import dansy
import random
import fusionClasses as fc
import EnrichementAnalysis

In [2]:
ref_df = dansy.import_proteome_files(ref_file_dir='./data/Current_Human_Proteome',
                                     ref_file_suffix='2026_0324.csv')
exon_information = pd.read_csv('Gene_exon_information.csv', index_col=0)
gene_conv = pd.read_csv('ENSEMBL_Gene_Conversion.csv')
valid_uniprots = list(set(ref_df['UniProt ID']).intersection(gene_conv['UniProtKB/Swiss-Prot ID'].unique()))

In [3]:
proteome_net = dansy.dansy(ref=ref_df, n=10)

Starting to fetch n-grams.
Finished getting all n-grams
Starting to generate adjacency
Finished building adjacency.


In [4]:
x = gene_conv.filter(['Gene stable ID', 'Gene name','UniProtKB/Swiss-Prot ID','Chromosome/scaffold name']).drop_duplicates()
conv_dict = x.set_index('UniProtKB/Swiss-Prot ID')['Gene stable ID'].to_dict()
name_conv = x.set_index('UniProtKB/Swiss-Prot ID')['Gene name'].to_dict()
chr_info = x.set_index('UniProtKB/Swiss-Prot ID')['Chromosome/scaffold name'].to_dict()

In [5]:
# Now let's take all the pairs and within each randomly choose exons to include
exons_grouped = exon_information.groupby('Gene stable ID')

In [8]:
# run through each pair and grab a random exon start point to call the breakpoint
natural_ngrams  = set(proteome_net.ngrams).union(proteome_net.collapsed_ngrams)
null_res_list = []
for i in range(100):
    random.seed(i*2)
    n = 15000 # How many pairs to create
    pairs = np.reshape(random.choices(valid_uniprots, k=n*2), (n,2)).tolist()
    null_fusions = []
    for pair in pairs:
        bp_info = pd.Series(index=['h_uniprot', 't_uniprot','h_pos','t_pos', 'h_chr', 't_chr', 'h_gene', 't_gene', 'h_symbol','t_symbol'],
                            dtype=object)
        bp_info['h_uniprot'] = pair[0]
        bp_info['t_uniprot'] = pair[1]
        bp_info['h_pos'] = random.choice(exons_grouped.get_group(conv_dict[pair[0]])['Exon region start (bp)'].values)
        bp_info['t_pos'] = random.choice(exons_grouped.get_group(conv_dict[pair[1]])['Exon region start (bp)'].values)
        bp_info['h_gene'] = conv_dict[pair[0]]
        bp_info['t_gene'] = conv_dict[pair[1]]
        bp_info['h_symbol'] = name_conv[pair[0]]
        bp_info['t_symbol'] = name_conv[pair[1]]
        bp_info['h_chr'] = chr_info[pair[0]]
        bp_info['t_chr'] = chr_info[pair[1]]
        null_fusions.append(bp_info)
        
    null_fusions = pd.DataFrame.from_records(null_fusions)
    null_fusions['name'] = null_fusions.h_symbol.str.cat(null_fusions.t_symbol, sep = '--')
    fusion_dansy = fc.fusionCollection(null_fusions, mapper={k:k for k in null_fusions.columns}, add_cols=[], silent=True)
    fusion_dansy.perform_dansy_analysis(dansy_obj=proteome_net)
    y = fusion_dansy.summarize_dansy_results()
    y['Novel_Architecture'] = y['dansy_impact'].apply(lambda x: str('Novel' in x))

    if i == 0:
        y.to_csv('260519_15K_Random_Pairs_DANSy_Results.csv')

    # Let's grab all the n-grams and all the domains from the natural proteome
    null_ngrams = [k for k in dansy.ngramUtilities.return_ngrams_from_list(y.domain_architecture.values,3)]
    
    # Now limiting them to only those found in the proteome
    null_ngrams = list(set(null_ngrams).intersection(natural_ngrams))
    null_enrichment_novel = EnrichementAnalysis.EnrichmentAnalysis(y,null_ngrams, {'category':'Novel_Architecture','domain_architecture':'domain_architecture'})
    null_res_list.append(null_enrichment_novel.res.set_index('ngram').True_q)

100%|██████████| 6472/6472 [00:10<00:00, 622.61it/s]


There were 2171 fusion domain architectures previously found in the proteome.


100%|██████████| 6530/6530 [00:11<00:00, 577.63it/s]


There were 2143 fusion domain architectures previously found in the proteome.


100%|██████████| 6422/6422 [00:10<00:00, 604.40it/s]


...

100%|██████████| 6516/6516 [00:07<00:00, 846.97it/s]


There were 2160 fusion domain architectures previously found in the proteome.


100%|██████████| 6483/6483 [00:07<00:00, 855.13it/s]


There were 2153 fusion domain architectures previously found in the proteome.


100%|██████████| 9063/9063 [00:32<00:00, 280.82it/s]


In [9]:
tcga_null_dists = pd.concat(null_res_list, axis = 1)
tcga_null_dists.to_csv('Null_p_val_dists_15K_tcga.csv')

Now repeating but with fewer pairs to recapture the cutoffs for the CCLE datasets instead

In [10]:
# run through each pair and grab a random exon start point to call the breakpoint
natural_ngrams  = set(proteome_net.ngrams).union(proteome_net.collapsed_ngrams)
null_res_list = []
for i in range(100):
    random.seed(i*2)
    n = 3000 # How many pairs to create
    pairs = np.reshape(random.choices(valid_uniprots, k=n*2), (n,2)).tolist()
    null_fusions = []
    for pair in pairs:
        bp_info = pd.Series(index=['h_uniprot', 't_uniprot','h_pos','t_pos', 'h_chr', 't_chr', 'h_gene', 't_gene', 'h_symbol','t_symbol'],
                            dtype=object)
        bp_info['h_uniprot'] = pair[0]
        bp_info['t_uniprot'] = pair[1]
        bp_info['h_pos'] = random.choice(exons_grouped.get_group(conv_dict[pair[0]])['Exon region start (bp)'].values)
        bp_info['t_pos'] = random.choice(exons_grouped.get_group(conv_dict[pair[1]])['Exon region start (bp)'].values)
        bp_info['h_gene'] = conv_dict[pair[0]]
        bp_info['t_gene'] = conv_dict[pair[1]]
        bp_info['h_symbol'] = name_conv[pair[0]]
        bp_info['t_symbol'] = name_conv[pair[1]]
        bp_info['h_chr'] = chr_info[pair[0]]
        bp_info['t_chr'] = chr_info[pair[1]]
        null_fusions.append(bp_info)
        
    null_fusions = pd.DataFrame.from_records(null_fusions)
    null_fusions['name'] = null_fusions.h_symbol.str.cat(null_fusions.t_symbol, sep = '--')
    fusion_dansy = fc.fusionCollection(null_fusions, mapper={k:k for k in null_fusions.columns}, add_cols=[],silent=True)
    fusion_dansy.perform_dansy_analysis(dansy_obj=proteome_net)
    y = fusion_dansy.summarize_dansy_results()
    y['Novel_Architecture'] = y['dansy_impact'].apply(lambda x: str('Novel' in x))
    # Let's grab all the n-grams and all the domains from the natural proteome
    null_ngrams = [k for k in dansy.ngramUtilities.return_ngrams_from_list(y.domain_architecture.values,3)]
    
    # Now limiting them to only those found in the proteome
    null_ngrams = list(set(null_ngrams).intersection(natural_ngrams))
    null_enrichment_novel = EnrichementAnalysis.EnrichmentAnalysis(y,null_ngrams, {'category':'Novel_Architecture','domain_architecture':'domain_architecture'})
    null_res_list.append(null_enrichment_novel.res.set_index('ngram').True_q)

100%|██████████| 1643/1643 [00:01<00:00, 944.62it/s]


There were 701 fusion domain architectures previously found in the proteome.


100%|██████████| 1627/1627 [00:01<00:00, 930.16it/s]


There were 699 fusion domain architectures previously found in the proteome.


100%|██████████| 1676/1676 [00:01<00:00, 908.97it/s]


...

100%|██████████| 1630/1630 [00:01<00:00, 937.16it/s]


There were 722 fusion domain architectures previously found in the proteome.


100%|██████████| 1635/1635 [00:02<00:00, 775.57it/s]


There were 719 fusion domain architectures previously found in the proteome.


100%|██████████| 3499/3499 [00:03<00:00, 1045.37it/s]


In [11]:
ccle_null_dists = pd.concat(null_res_list, axis = 1)
ccle_null_dists.to_csv('Null_p_val_dists_3K_ccle.csv')